# Detecting Data Drift with Rule-Based Checks

See [`docs/04-detecting-data-drift.md`](../docs/04-detecting-data-drift.md#rule-based-checks) for the
concept notes this notebook is based on.

Rule-based checks don't measure drift magnitude at all — they're straightforward, explicit
thresholds that serve as cheap, always-on alerting heuristics: "notify me whenever X." This notebook
builds a tiny rule engine and runs it against the same scenario as the other three notebooks.

All four notebooks in this folder analyze the exact same synthetic `reference`/`current` pair from
[`src/drift_dataset.py`](../src/drift_dataset.py), so results are directly comparable across techniques.

## Load the shared reference/current dataset

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import (
    CATEGORICAL_COLUMNS,
    COLUMN_DRIFT_GROUND_TRUTH,
    NUMERICAL_COLUMNS,
    make_reference_and_current,
)

pd.set_option("display.width", 120)

reference, current = make_reference_and_current(seed=42)
print("reference:", reference.shape, " current:", current.shape)
reference.head()

reference: (5000, 6)  current: (2000, 6)


,channel,product_category,basket_size,discount_pct,unit_price,customer_age
0,in_store,electronics,21.102831,0.031317,17.104594,32.639729
1,in_store,grocery,63.178040,0.062780,22.418914,46.994313
2,online,home,51.468098,0.173390,15.736280,43.558738
3,in_store,electronics,46.992552,0.189024,16.649926,52.330196
4,in_store,electronics,53.157178,0.080097,18.866577,44.449209


## A minimal rule engine

Each rule is just a name, a severity, and a function that inspects `reference`/`current` and returns
`(triggered, detail)`.

In [2]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class Rule:
    name: str
    severity: str
    check: Callable[[pd.DataFrame, pd.DataFrame], tuple]

def run_rules(rules, ref, cur):
    rows = []
    for rule in rules:
        triggered, detail = rule.check(ref, cur)
        rows.append({
            "rule": rule.name,
            "severity": rule.severity,
            "triggered": triggered,
            "detail": detail,
        })
    return pd.DataFrame(rows)

## Rule definitions

Four rules, each mirroring an example from the docs, adapted to this dataset:

1. **Category share threshold** — alert if the `online` channel share exceeds 40%.
2. **New category appeared** — alert if `product_category` contains a value not seen in `reference`.
3. **Out-of-range share** — alert if more than 1% of `customer_age` values fall outside a plausible
   `[18, 90]` range.
4. **Mean-shift threshold** — alert if `basket_size`'s current mean falls outside a
   `reference_mean ± 2 * reference_std` band (the same two-sigma idea from notebook 01, framed here
   as an explicit rule).

In [3]:
def check_channel_share(ref, cur, channel="online", threshold=0.40):
    share = (cur["channel"] == channel).mean()
    return share > threshold, f"{channel} share = {share:.1%} (threshold {threshold:.0%})"

def check_new_category(ref, cur, column="product_category"):
    new_categories = sorted(set(cur[column]) - set(ref[column]))
    detail = f"new categories: {new_categories}" if new_categories else "no new categories"
    return len(new_categories) > 0, detail

def check_out_of_range(ref, cur, column="customer_age", lo=18, hi=90, threshold=0.01):
    share = ((cur[column] < lo) | (cur[column] > hi)).mean()
    return share > threshold, f"{share:.2%} of values outside [{lo}, {hi}] (threshold {threshold:.0%})"

def check_mean_shift(ref, cur, column="basket_size", n_std=2):
    lo = ref[column].mean() - n_std * ref[column].std()
    hi = ref[column].mean() + n_std * ref[column].std()
    cur_mean = cur[column].mean()
    triggered = not (lo <= cur_mean <= hi)
    return triggered, f"current mean {cur_mean:.2f} vs. reference band [{lo:.2f}, {hi:.2f}]"

rules = [
    Rule("online channel share > 40%", "high", check_channel_share),
    Rule("new product_category value appears", "high", check_new_category),
    Rule("customer_age out-of-range share > 1%", "medium", check_out_of_range),
    Rule("basket_size mean outside 2-sigma band", "medium", check_mean_shift),
]

report = run_rules(rules, reference, current)
report

,rule,severity,triggered,detail
0,online channel share > 40%,high,True,online share = 44.5% (threshold 40%)
1,new product_category value appears,high,True,new categories: ['online_exclusive']
2,customer_age out-of-range share > 1%,medium,True,"1.25% of values outside [18, 90] (threshold 1%)"
3,basket_size mean outside 2-sigma band,medium,False,"current mean 57.92 vs. reference band [14.62, ..."


In [4]:
triggered = report[report["triggered"]]
print(f"{len(triggered)} of {len(report)} rules triggered:")
for _, row in triggered.iterrows():
    print(f"  [{row['severity'].upper()}] {row['rule']} -- {row['detail']}")

3 of 4 rules triggered:
  [HIGH] online channel share > 40% -- online share = 44.5% (threshold 40%)
  [HIGH] new product_category value appears -- new categories: ['online_exclusive']
  [MEDIUM] customer_age out-of-range share > 1% -- 1.25% of values outside [18, 90] (threshold 1%)


## Cross-checking against notebooks 02 and 03

Three of the four rules triggered, and none of them computed a p-value or a distance — just a
threshold — yet they flag the same real issues the statistical tests and distance metrics found:
the channel share shift, the new `online_exclusive` category, and the `customer_age` data-quality
bug.

The fourth rule is the more interesting result: the **basket_size mean-shift rule did not trigger**,
even though `basket_size` is the most clearly drifted feature in this whole scenario — flagged
strongly by both the KS test and the Wasserstein/PSI metrics in notebooks 02 and 03. This is the
exact same blind spot found in `01_summary_statistics.ipynb`: `basket_size`'s reference data has
enough natural spread that a plain ±2-sigma threshold isn't tight enough to catch a shift of this
size. A rule is only as good as the threshold it was written with — unlike a statistical test or
distance metric, it won't automatically adapt its sensitivity to the feature's own variance.

`discount_pct`'s small, borderline drift (the one that only became "significant" at large sample
sizes in notebook 02) has **no rule at all** here, since nobody wrote a threshold for it — a second,
simpler illustration of rule-based checks' main limitation: they only catch what you thought to
write a rule for, and only as precisely as that rule was tuned.

## When to use rule-based checks

- **Cheapest possible always-on alerting.** No distributional assumptions, no p-values to
  interpret — just a threshold a non-technical stakeholder can read directly ("share of fraud
  predictions is over 10%").
- Good for things you already know are meaningful: a brand-new category value, an impossible value
  range, a known-important feature's share crossing a business-defined line.
- **Blind spot**: rules only catch what you explicitly wrote a check for. They don't generalize to
  "something changed" the way a statistical test or distance metric does — pair them with one of
  those (`02_statistical_tests.ipynb`, `03_distance_metrics.ipynb`) for broader coverage, and use
  summary statistics (`01_summary_statistics.ipynb`) to decide which thresholds are worth setting in
  the first place.

## Answer key

What this synthetic scenario was built to demonstrate for each column:

In [5]:
pd.Series(COLUMN_DRIFT_GROUND_TRUTH, name="ground_truth").to_frame()

,ground_truth
channel,drift
product_category,drift
basket_size,drift
discount_pct,borderline
unit_price,none
customer_age,data_quality
